In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# ================= 配置区域 =================
DATA_DIR = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal"
# LUT_PATH 暂时不需要了，因为移除了 Label Overlay，但保留变量以防万一
LUT_PATH = "Freesurfer_LUT_alex_labels_jiayi.xlsx"

# 通道定义
CH_UFA    = 14
CH_AMIDE  = 225
CH_MPRAGE = 341
CH_QSM    = 350  # 新增: QSM (第351维 -> index 350)

# ================= 绘图参数配置 =================
TARGET_AXIS = 'Sagittal'   # 'Axial' | 'Coronal' | 'Sagittal'
ROTATION_K  = 1            # 保持原有逻辑: 左转90°（逆时针）
NUM_IMAGES  = 5
SLICE_STEP  = 3

# 论文友好字体参数
TITLE_FS = 16  # Panel 标题字号
CBAR_FS  = 13  # Colorbar label 字号
TICK_FS  = 11  # Colorbar tick 字号

def normalize_robust(img, p_low=1, p_high=99):
    """鲁棒归一化，防止 QSM/CEST 的极端噪点破坏对比度"""
    if np.all(img == 0): return img
    vmin, vmax = np.percentile(img, [p_low, p_high])
    return np.clip(img, vmin, vmax)

def auto_find_best_range(labels_vol, axis, num_imgs, step):
    """(保持不变) 根据每层非零 label 数量，自动找最大脑面积的中心切片"""
    print(f"正在扫描 {axis} 方向的大脑截面大小...")

    if axis == 'Axial':       # Z
        area_per_slice = np.count_nonzero(labels_vol, axis=(0, 1))
    elif axis == 'Coronal':   # Y
        area_per_slice = np.count_nonzero(labels_vol, axis=(0, 2))
    else:                     # Sagittal: X
        area_per_slice = np.count_nonzero(labels_vol, axis=(1, 2))

    center_idx = int(np.argmax(area_per_slice))
    max_area = int(area_per_slice[center_idx])

    if max_area < 100:
        print("⚠️ 警告：检测到的脑区面积很小，可能是 label 为空或方向/维度不匹配。")

    print(f"检测到最大截面位于层 {center_idx} (Area: {max_area} voxels)")

    half_span = (num_imgs * step) // 2
    start = max(0, center_idx - half_span)
    end = min(len(area_per_slice), center_idx + half_span + step)

    slice_indices = list(range(start, end, step))[:num_imgs]
    return slice_indices

def get_slice_corrected(data, labels, axis, idx, k_rot):
    """(保持不变) 提取切片并旋转修正"""
    if axis == 'Axial':
        sl_d = data[:, :, idx, :]   # (X,Y,C)
        sl_l = labels[:, :, idx]    # (X,Y)
    elif axis == 'Coronal':
        sl_d = data[:, idx, :, :]   # (X,Z,C)
        sl_l = labels[:, idx, :]    # (X,Z)
    else:  # Sagittal
        sl_d = data[idx, :, :, :]   # (Y,Z,C)
        sl_l = labels[idx, :, :]    # (Y,Z)

    # 旋转修正
    sl_d = np.rot90(sl_d, k=k_rot)
    sl_l = np.rot90(sl_l, k=k_rot)

    return sl_d, sl_l

def plot_contrasts_2x2(img_mprage, img_qsm, img_ufa, img_amide,
                       title_fs=16, cbar_fs=13, tick_fs=11):
    """
    修改后的 2x2 面板：展示四种不同的模态对比度
    A: MPRAGE (Structural)
    B: QSM (Susceptibility)
    C: QTI (Microstructure)
    D: CEST (Metabolic)
    """
    fig = plt.figure(figsize=(12, 11), dpi=300, constrained_layout=True)

    # 保持原有的网格比例，确保完美对齐
    gs = fig.add_gridspec(
        2, 4,
        width_ratios=[1.0, 0.05, 1.0, 0.05], # 主图:Cbar:主图:Cbar
        wspace=0.05,
        hspace=0.05
    )

    # Row 1
    axA  = fig.add_subplot(gs[0, 0]); caxA = fig.add_subplot(gs[0, 1])
    axB  = fig.add_subplot(gs[0, 2]); caxB = fig.add_subplot(gs[0, 3])
    # Row 2
    axC  = fig.add_subplot(gs[1, 0]); caxC = fig.add_subplot(gs[1, 1])
    axD  = fig.add_subplot(gs[1, 2]); caxD = fig.add_subplot(gs[1, 3])

    # --- Panel A: MPRAGE (无需 Colorbar) ---
    axA.imshow(normalize_robust(img_mprage), cmap='gray', aspect='equal')
    axA.set_title(r"$\bf{(A)}$ MPRAGE (Structural)", loc='left', fontsize=title_fs, pad=8)
    axA.axis('off')
    caxA.axis('off') # 占位隐藏，保持对齐

    # --- Panel B: QSM (新增, 需 Colorbar) ---
    # QSM 通常有正负值 (Paramagnetic/Diamagnetic)，这里用 gray 或 bone
    imB = axB.imshow(normalize_robust(img_qsm), cmap='gray', aspect='equal')
    axB.set_title(r"$\bf{(B)}$ QSM (Susceptibility)", loc='left', fontsize=title_fs, pad=8)
    axB.axis('off')
    # 激活 caxB
    cbB = fig.colorbar(imB, cax=caxB)
    cbB.set_label("Susceptibility (ppm)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbB.ax.tick_params(labelsize=tick_fs)

    # --- Panel C: QTI uFA ---
    imC = axC.imshow(normalize_robust(img_ufa), cmap='gray', aspect='equal')
    axC.set_title(r"$\bf{(C)}$ QTI Microstructure ($\mu$FA)", loc='left', fontsize=title_fs, pad=8)
    axC.axis('off')
    cbC = fig.colorbar(imC, cax=caxC)
    cbC.set_label("$\mu$FA (a.u.)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbC.ax.tick_params(labelsize=tick_fs)

    # --- Panel D: CEST Amide ---
    # 使用 nearest 插值保留低分辨率特征
    imD = axD.imshow(normalize_robust(img_amide), cmap='gray', aspect='equal', interpolation='nearest')
    axD.set_title(r"$\bf{(D)}$ CEST Amide (Low-Res)", loc='left', fontsize=title_fs, pad=8)
    axD.axis('off')
    cbD = fig.colorbar(imD, cax=caxD)
    cbD.set_label("Amide Contrast (a.u.)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbD.ax.tick_params(labelsize=tick_fs)

    plt.show()

# ================= 主执行逻辑 =================
mat_files = sorted(list(Path(DATA_DIR).glob("*.mat")))
if not mat_files:
    raise FileNotFoundError(f"在 {DATA_DIR} 未找到 .mat 文件，请检查路径。")

target_file = mat_files[0]
print(f"📂 正在加载数据: {target_file.name} ...")

with h5py.File(target_file, "r") as f:
    raw_data = f["data"][:]  
    data_vol = np.moveaxis(raw_data, 0, -1) 

    labels_vol = f["region_labels"][:]
    if labels_vol.ndim == 4 and labels_vol.shape[0] == 1:
        labels_vol = labels_vol[0, ...] 
    elif labels_vol.ndim == 4:
        labels_vol = np.moveaxis(labels_vol, 0, -1)

print(f"✅ Data Shape:    {data_vol.shape}")


In [ ]:

# 选层
target_slices = auto_find_best_range(labels_vol, TARGET_AXIS, NUM_IMAGES, SLICE_STEP)
print(f"将在以下层生成多模态对比图: {target_slices}")

# 循环画图
for slice_idx in target_slices:
    try:
        sl_data, sl_label = get_slice_corrected(data_vol, labels_vol, TARGET_AXIS, slice_idx, ROTATION_K)

        img_mprage = sl_data[..., CH_MPRAGE]
        img_ufa    = sl_data[..., CH_UFA]
        img_amide  = sl_data[..., CH_AMIDE]
        img_qsm    = sl_data[..., CH_QSM] # 提取 QSM

        plot_contrasts_2x2(
            img_mprage, img_qsm, img_ufa, img_amide,
            title_fs=TITLE_FS, cbar_fs=CBAR_FS, tick_fs=TICK_FS
        )

    except IndexError as e:
        print(f"层 {slice_idx} 读取出错: {e}")
        continue

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# ================= 配置区域 =================
DATA_DIR = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal"
LUT_PATH = "Freesurfer_LUT_alex_labels_jiayi.xlsx"

# 通道定义 (已修正：文档序号 - 1)
CH_UFA    = 14   # 文档第 15
CH_AMIDE  = 225  # 文档第 226
CH_MPRAGE = 341  # 文档第 342
CH_QSM    = 350  # 文档第 351 (最后一个)

# ================= 绘图参数 =================
TARGET_AXIS = 'Sagittal'
ROTATION_K  = 1
NUM_IMAGES  = 5
SLICE_STEP  = 3
TITLE_FS = 16
CBAR_FS  = 13
TICK_FS  = 11

def normalize_robust(img, p_low=1, p_high=99):
    if np.all(img == 0): return img
    vmin, vmax = np.percentile(img, [p_low, p_high])
    return np.clip(img, vmin, vmax)

def auto_find_best_range(labels_vol, axis, num_imgs, step):
    print(f"正在扫描 {axis} 方向的大脑截面大小...")
    if axis == 'Axial':     area = np.count_nonzero(labels_vol, axis=(0, 1))
    elif axis == 'Coronal': area = np.count_nonzero(labels_vol, axis=(0, 2))
    else:                   area = np.count_nonzero(labels_vol, axis=(1, 2))
    
    center_idx = int(np.argmax(area))
    start = max(0, center_idx - (num_imgs * step) // 2)
    end = start + num_imgs * step
    return list(range(start, end, step))[:num_imgs]

def get_slice_corrected(data, labels, axis, idx, k_rot):
    if axis == 'Axial':   d, l = data[:,:,idx,:], labels[:,:,idx]
    elif axis == 'Coronal': d, l = data[:,idx,:,:], labels[:,idx,:]
    else:                 d, l = data[idx,:,:,:], labels[idx,:,:]
    return np.rot90(d, k=k_rot), np.rot90(l, k=k_rot)

def plot_contrasts_2x2(img_mprage, img_qsm, img_ufa, img_amide,
                       title_fs=16, cbar_fs=13, tick_fs=11):
    fig = plt.figure(figsize=(12, 11), dpi=300, constrained_layout=True)
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 0.05, 1, 0.05], wspace=0.05, hspace=0.05)

    # Row 1
    axA = fig.add_subplot(gs[0,0]); caxA = fig.add_subplot(gs[0,1])
    axB = fig.add_subplot(gs[0,2]); caxB = fig.add_subplot(gs[0,3])
    # Row 2
    axC = fig.add_subplot(gs[1,0]); caxC = fig.add_subplot(gs[1,1])
    axD = fig.add_subplot(gs[1,2]); caxD = fig.add_subplot(gs[1,3])

    # A: MPRAGE
    axA.imshow(normalize_robust(img_mprage), cmap='gray', aspect='equal')
    axA.set_title(r"$\bf{(A)}$ MPRAGE (Structural)", loc='left', fontsize=title_fs, pad=8)
    axA.axis('off'); caxA.axis('off')

    # B: QSM
    imB = axB.imshow(normalize_robust(img_qsm), cmap='gray', aspect='equal')
    axB.set_title(r"$\bf{(B)}$ QSM (Susceptibility)", loc='left', fontsize=title_fs, pad=8)
    axB.axis('off')
    cbB = fig.colorbar(imB, cax=caxB)
    cbB.set_label("Susceptibility (ppm)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbB.ax.tick_params(labelsize=tick_fs)

    # C: QTI
    imC = axC.imshow(normalize_robust(img_ufa), cmap='gray', aspect='equal')
    axC.set_title(r"$\bf{(C)}$ QTI Microstructure ($\mu$FA)", loc='left', fontsize=title_fs, pad=8)
    axC.axis('off')
    cbC = fig.colorbar(imC, cax=caxC)
    cbC.set_label("$\mu$FA (a.u.)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbC.ax.tick_params(labelsize=tick_fs)

    # D: CEST
    imD = axD.imshow(normalize_robust(img_amide), cmap='gray', aspect='equal', interpolation='nearest')
    axD.set_title(r"$\bf{(D)}$ CEST Amide (Low-Res)", loc='left', fontsize=title_fs, pad=8)
    axD.axis('off')
    cbD = fig.colorbar(imD, cax=caxD)
    cbD.set_label("Amide Contrast (a.u.)", rotation=270, labelpad=18, fontsize=cbar_fs)
    cbD.ax.tick_params(labelsize=tick_fs)
    
    plt.show()

# ================= 主程序 =================
mat_files = sorted(list(Path(DATA_DIR).glob("*.mat")))
if not mat_files: raise FileNotFoundError("未找到数据文件")

target_file = mat_files[0]
print(f"📂 加载数据: {target_file.name}")

with h5py.File(target_file, "r") as f:
    raw_data = f["data"][:]
    data_vol = np.moveaxis(raw_data, 0, -1)
    
    # ----------------------------------------------------
    # 🔥 关键检查：验证通道数
    # ----------------------------------------------------
    n_channels = data_vol.shape[-1]
    print(f"✅ 数据加载完成。维度: {data_vol.shape}")
    print(f"✅ 总通道数: {n_channels}")
    
    if n_channels == 351:
        print("✅ 验证通过：通道数与描述 (351) 完全一致！")
        print("   -> Index 350 是最后一个通道 (QSM)，配置正确。")
    else:
        print(f"⚠️ 警告：数据通道数是 {n_channels}，而描述是 351。请再次检查维度！")

    labels_vol = f["region_labels"][:]
    if labels_vol.ndim == 4:
        if labels_vol.shape[0] == 1: labels_vol = labels_vol[0]
        else: labels_vol = np.moveaxis(labels_vol, 0, -1)

# 绘图
target_slices = auto_find_best_range(labels_vol, TARGET_AXIS, NUM_IMAGES, SLICE_STEP)
for slice_idx in target_slices:
    try:
        sl_data, _ = get_slice_corrected(data_vol, labels_vol, TARGET_AXIS, slice_idx, ROTATION_K)
        # 这里使用了修正后的 0-based 索引
        plot_contrasts_2x2(
            sl_data[..., CH_MPRAGE], 
            sl_data[..., CH_QSM], 
            sl_data[..., CH_UFA], 
            sl_data[..., CH_AMIDE],
            title_fs=TITLE_FS, cbar_fs=CBAR_FS, tick_fs=TICK_FS
        )
    except IndexError as e:
        print(f"❌ 索引错误 (层 {slice_idx}): {e}")